# 3. Pupils and point-spread functions

Telescope pupils are anti-aliased masks; the point-spread function (PSF) is the intensity in the focal
plane. The PSF of a uniform circular pupil is the Airy pattern, with its first dark ring at
$1.22\,\lambda/D$ (Born & Wolf, *Principles of Optics*, §8.5.2).

In [ ]:
from astropy import units as u

import lumos
from lumos import plotting
from lumos.elements import apertures
from lumos.fields import Grid, Wavefront
from lumos.plotting import style

style.apply()

In [ ]:
fig, axes = style.figure("double", ncols=3, aspect=0.33)
for ax, pupil, size in zip(
    axes, [apertures.vlt(), apertures.jwst(), apertures.elt()], [8.6, 6.8, 40.0], strict=True
):
    grid = Grid(256, dx=size / 256 * u.m)
    plane = Wavefront.plane(grid, 1 * u.um, power=1 * u.W)
    plotting.plot_intensity(pupil.apply(**{"in": plane})["out"], ax=ax, colorbar=False)
    ax.set_title(pupil.name)

## The PSF of the VLT

The preset `vlt_psf.lumos` observes a star in the H band (three spectral channels) with an 8.2 m telescope.

In [ ]:
ctx = lumos.io.load("presets/vlt_psf.lumos")
result = ctx.run(ctx.meta["outputs"], archive="vlt-psf")
psf = result["image.out"]
fig, axes = style.figure("double", ncols=2, aspect=0.45)
plotting.plot_psf(psf, ax=axes[0], diameter=8.2 * u.m, focal_length=120 * u.m)
plotting.plot_radial_profile(psf, ax=axes[1])
result.archive.save_figure(fig, "vlt_psf");

## A coronagraph

The preset `coronagraph.lumos` places a companion 10⁴ times fainter than its star at 6 λ/D behind a
charge-2 vortex coronagraph (Mawet et al. 2005): the star is removed and the companion appears.

In [ ]:
ctx = lumos.io.load("presets/coronagraph.lumos")
result = ctx.run(ctx.meta["outputs"], archive="coronagraph")
before, after = result["telescope.out"].power, result["coronagraph.out"].power
fig, ax = style.figure("single")
plotting.plot_psf(result["image.out"], ax=ax, diameter=8 * u.m, focal_length=100 * u.m)
result.archive.save_figure(fig, "coronagraphic_image")
(after / before).decompose()